# Training Loop

训练循环是深度学习的"日常操作"：数据划分、mini-batch、shuffle、epoch、验证。本课把完整训练流程拆解成规范骨架。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 数据划分：train / val / test


- **train**：更新参数
- **validation**：选超参数、监控过拟合（训练时反复查看，会"间接拟合"它）
- **test**：只评估一次，最终汇报泛化能力

常见比例 8:1:1。**严禁**用 test 调参——那会让 test 变成 val 的延长。


In [ ]:
rng = np.random.default_rng(42)
n = 400
X0 = rng.standard_normal((n, 2)) + np.array([-2.0, 0.0])
X1 = rng.standard_normal((n, 2)) + np.array([2.0, 0.0])
X = np.vstack([X0, X1]); y = np.concatenate([np.zeros(n), np.ones(n)])

idx = rng.permutation(len(X))
n_tr, n_va = int(0.7*len(X)), int(0.15*len(X))
tr, va, te = idx[:n_tr], idx[n_tr:n_tr+n_va], idx[n_tr+n_va:]
print(f"train {len(tr)} / val {len(va)} / test {len(te)}")


## 2. mini-batch 与 shuffle


- **batch**：一次梯度更新用的样本数。全量梯度（batch=全部）稳定但慢、易困在平坦区；单样本 SGD 噪声大；mini-batch 折中
- **shuffle**：打乱样本顺序，让每个 batch 的梯度估计接近"真实期望梯度"。**不 shuffle 时**，按类别排序的数据会让每个 batch 只含一类，梯度来回振荡


In [ ]:
def make_batches(X, y, bs, shuffle=True, seed=0):
    r = np.random.default_rng(seed)
    idx = np.arange(len(X))
    if shuffle:
        r.shuffle(idx)
    for i in range(0, len(idx), bs):
        sel = idx[i:i+bs]
        yield X[sel], y[sel]

def train_with_shuffle(X, y, shuffle, seed=0):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    xt = torch.tensor(X, dtype=torch.float32); yt = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    losses = []
    for epoch in range(60):
        ep_loss = []
        for xb, yb in make_batches(X, y, 32, shuffle=shuffle, seed=seed):
            opt.zero_grad()
            xbt = torch.tensor(xb, dtype=torch.float32); ybt = torch.tensor(yb, dtype=torch.float32).unsqueeze(1)
            loss = F.binary_cross_entropy(torch.sigmoid(model(xbt)), ybt)
            loss.backward(); opt.step()
            ep_loss.append(loss.item())
        losses.append(np.mean(ep_loss))
    return losses

l_shuf = train_with_shuffle(X, y, True)
l_noshuf = train_with_shuffle(X, y, False)

plt.figure(figsize=(8, 4.5))
plt.plot(l_shuf, label='shuffle=True')
plt.plot(l_noshuf, label='shuffle=False')
plt.xlabel('epoch'); plt.ylabel('平均 batch 损失')
plt.title('shuffle 对收敛的影响（数据按类别顺序排列时尤其明显）')
plt.legend(); plt.grid(alpha=0.3)


## 3. 完整训练骨架


标准流程：每个 epoch 内，遍历 batches → 前向 → 算损失 → `zero_grad` → `backward` → `step`；epoch 末在 val 上评估。`zero_grad` 必不可少——梯度默认**累加**。


In [ ]:
def train(model, opt, Xtr, ytr, Xva, yva, epochs=100, bs=32, seed=0):
    tr_loss, va_loss, va_acc = [], [], []
    xtr = torch.tensor(Xtr, dtype=torch.float32); ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
    xva = torch.tensor(Xva, dtype=torch.float32); yva_t = torch.tensor(yva, dtype=torch.float32).unsqueeze(1)
    for epoch in range(epochs):
        model.train()
        ep = []
        for xb, yb in make_batches(xtr, ytr_t, bs, seed=seed+epoch):
            opt.zero_grad()
            loss = F.binary_cross_entropy(torch.sigmoid(model(xb)), yb)
            loss.backward(); opt.step()
            ep.append(loss.item())
        model.eval()
        with torch.no_grad():
            vl = F.binary_cross_entropy(torch.sigmoid(model(xva)), yva_t).item()
            acc = ((torch.sigmoid(model(xva)).ravel() > 0.5) == yva_t.ravel()).float().mean().item()
        tr_loss.append(np.mean(ep)); va_loss.append(vl); va_acc.append(acc)
    return tr_loss, va_loss, va_acc

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
opt = torch.optim.SGD(model.parameters(), lr=0.1)
tr_l, va_l, va_a = train(model, opt, X[tr], y[tr], X[va], y[va])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(tr_l, label='train'); axes[0].plot(va_l, label='val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('BCE loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(va_a); axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val 准确率'); axes[1].grid(alpha=0.3)
axes[0].set_title('训练/验证损失'); axes[1].set_title('验证准确率')
plt.tight_layout()


## 4. 训练中的常见问题速查


| 现象 | 常见原因 | 检查方法 |
|------|----------|----------|
| loss 为 NaN | 学习率过大 / log 里出现 0 | 打印梯度范数、减小 lr |
| loss 不下降 | 学习率太小 / 数据未归一化 | 曲线放大看、调 lr |
| train 好 val 差 | 过拟合 | 看 train/val 差距（07 课） |
| 振荡剧烈 | batch 太小 / lr 大 | 增大 batch、加动量 |
| 梯度不更新 | zero_grad 缺失 / requires_grad=False | 检查 `w.grad` 是否 None |


## 课后练习


1. **改 batch**：把 batch size 从 32 改成 8 与 128，比较损失曲线的噪声与收敛速度。
2. **改 lr**：在 [0.01, 0.05, 0.5, 2.0] 中对比，找出梯度下降的"甜点区间"。
3. **实现 Early Stop**：val 损失连续 N 轮不降就停止，返回最优权重。
4. **数据归一化**：把 X 缩放 100 倍后训练，观察损失曲线变化（呼应 05 课初始化）。
5. **思考**：为什么测试集只能看一次？如果反复用 test 调参会发生什么？
